In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn

In [2]:
DATA = Path("../../data/")
DATA_PROCESSED = DATA / "processed"
DATA_PREDICTIONS = DATA / "predictions"

train_returns = pd.read_csv(DATA_PROCESSED / "train_returns.csv", index_col=0, parse_dates=True)
valid_returns = pd.read_csv(DATA_PROCESSED / "valid_returns.csv", index_col=0, parse_dates=True)
test_returns = pd.read_csv(DATA_PROCESSED / "test_returns.csv", index_col=0, parse_dates=True)

ASSETS = train_returns.columns.tolist()
NUM_ASSETS = len(ASSETS)

In [3]:
for name, df in [
    ("Train", train_returns),
    ("Valid", valid_returns),
    ("Test", test_returns)
]:
    assert df.columns.tolist() == ASSETS
    assert np.isfinite(df.to_numpy()).all()

    print(
        f"{name:12s} | "
        f"{df.index.min().date()} - {df.index.max().date()} | "
        f"{df.shape}"
    )

Train        | 2015-05-14 - 2017-12-29 | (646, 30)
Valid        | 2018-01-02 - 2018-12-28 | (243, 30)
Test         | 2019-01-02 - 2024-08-30 | (1376, 30)


In [4]:
class BaselineModel:
    def __init__(self, num_assets: int, window_size: int = 1, window_type: str = "rolling"):
        assert window_size > 0
        assert window_type in ("rolling", "expanding")

        self.num_assets = num_assets
        self.window_size = window_size
        self.window_type = window_type

    def forward(self, x: np.ndarray) -> np.ndarray:
        # x: (days, num_assets)
        # out: (days - window_size + 1, num_assets)
        assert x.ndim == 2
        assert x.shape[1] == self.num_assets
        assert x.shape[0] >= self.window_size

        out = []
        for i in range(x.shape[0] - self.window_size + 1):
            end = i + self.window_size
            inputs = x[i:end] if self.window_type == "rolling" else x[:end]
            out.append(np.mean(inputs, axis=0))

        return np.stack(out, axis=0)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

# Training and Evaluation

In [5]:
# No training is required for the historical mean baseline

In [6]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    assert y_true.shape == y_pred.shape
    assert y_true.ndim == 2
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    errors = y_true - y_pred

    mse = np.mean(np.power(errors, 2))
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

    true_df = pd.DataFrame(y_true)
    pred_df = pd.DataFrame(y_pred)
    ic = pred_df.corrwith(true_df, axis=1, method="pearson")
    rank_ic = pred_df.corrwith(true_df, axis=1, method="spearman")

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Directional Accuracy": directional_accuracy,
        "IC": ic.mean(),
        "Rank IC": rank_ic.mean(),
        "ICIR": ic.mean() / ic.std(),
        "Rank ICIR": rank_ic.mean() / rank_ic.std()
    }

# Hyperparameter tuning

In [7]:
best_ic = -float("inf")
best_config = None

for window_type in ("rolling", "expanding"):
    for window_size in (1, 2, 4, 8, 16, 32, 64, 128, 7, 14, 21, 28):
        model = BaselineModel(num_assets=NUM_ASSETS, window_size=window_size, window_type=window_type)

        history = pd.concat([train_returns.tail(window_size), valid_returns])
        X_valid = history.iloc[:-1].to_numpy()
        y_valid_true = valid_returns.to_numpy()
        y_valid_pred = model(X_valid)

        valid_metrics = evaluate(y_valid_true, y_valid_pred)

        if valid_metrics["IC"] > best_ic:
            best_ic = valid_metrics["IC"]

            best_config = {
                "window_type": window_type,
                "window_size": window_size,
                "best_metric": best_ic.item(),
            }

display(best_config)

{'window_type': 'rolling',
 'window_size': 128,
 'best_metric': 0.010089216584001031}

# Out-Of-Sample

In [8]:
model = BaselineModel(NUM_ASSETS, window_size=best_config["window_size"], window_type=best_config["window_type"])

history = pd.concat([train_returns.tail(model.window_size), valid_returns, test_returns])
X_test = history.iloc[:-1].to_numpy()
y_test_true = test_returns.to_numpy()
y_test_pred = model(X_test)[-len(test_returns):]
test_metrics = evaluate(y_test_true, y_test_pred)
test_metrics

{'MSE': np.float64(0.0006661421482608419),
 'RMSE': np.float64(0.025809729720801842),
 'MAE': np.float64(0.017927439677325182),
 'Directional Accuracy': np.float64(0.47170542635658913),
 'IC': np.float64(-0.004681199172487382),
 'Rank IC': np.float64(-0.021939716231528197),
 'ICIR': np.float64(-0.01536757505217938),
 'Rank ICIR': np.float64(-0.08473920071080743)}

## Saving

In [9]:
test_predictions = pd.DataFrame(y_test_pred, index=test_returns.index, columns=test_returns.columns)
test_predictions.to_csv(DATA_PREDICTIONS / "baseline_oos_predictions.csv")
display(test_predictions)

,600169.SH,000049.SZ,600428.SH,002064.SZ,000501.SZ,603000.SH,000712.SZ,600580.SH,600017.SH,603355.SH,...,600718.SH,600507.SH,002583.SZ,601718.SH,600835.SH,002268.SZ,002092.SZ,600098.SH,002340.SZ,300001.SZ
date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,0.000099,0.000759,-0.000247,0.000017,-0.001719,-0.000569,0.000185,-0.000980,-0.000898,-0.001414,...,-0.000153,-0.000791,0.000627,-0.000660,-0.001363,0.000099,-0.002446,-0.000704,-0.003130,0.003943
2019-01-03,0.000065,0.000747,-0.000269,0.000054,-0.002040,-0.000537,0.000116,-0.001073,-0.001068,-0.001243,...,-0.000385,-0.000768,0.000497,-0.000778,-0.001472,-0.000097,-0.002393,-0.000684,-0.003296,0.003758
2019-01-04,-0.000003,0.000808,-0.000130,0.000531,-0.001792,-0.000291,0.000284,-0.001081,-0.001006,-0.001198,...,-0.000688,-0.000197,0.000307,-0.000655,-0.001531,-0.000203,-0.002255,-0.000826,-0.003153,0.003553
2019-01-07,0.000204,0.000924,0.000061,0.000642,-0.001560,0.000097,0.000989,-0.000788,-0.000838,-0.001230,...,-0.000450,0.000082,0.000745,-0.000430,-0.001207,-0.000098,-0.001810,-0.000697,-0.002838,0.003848
2019-01-08,0.000271,0.000653,0.000020,0.000580,-0.001571,-0.000055,0.001005,-0.000744,-0.000818,-0.001498,...,-0.000722,0.000209,0.000333,-0.000282,-0.001238,-0.000336,-0.002006,-0.000818,-0.002973,0.003780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-26,-0.001537,0.000966,0.000593,0.000763,-0.000644,-0.001859,0.000281,0.000684,0.000363,0.000599,...,0.000078,-0.001117,-0.000701,-0.001069,-0.000065,-0.003320,-0.002761,0.001064,0.001220,-0.000356
2024-08-27,-0.001338,0.001299,0.000740,0.000776,-0.000552,-0.002622,-0.000116,0.000902,0.000306,0.000480,...,0.000089,-0.001011,-0.000762,-0.000975,-0.000051,-0.003311,-0.002648,0.001052,0.001522,-0.000215
2024-08-28,-0.001423,0.000745,0.000651,0.000899,-0.000623,-0.002413,0.000009,0.000562,0.000306,0.000366,...,0.000016,-0.001014,-0.001054,-0.001064,-0.000181,-0.003590,-0.002744,0.001024,0.001387,-0.000593
